# WikiProject Medicine — Overview

In this project, you are acting as **data scientists for the Wikimedia Foundation**.

You have **three datasets** about medical articles on Wikipedia:

1. **Edits**: who changed what, where and when.
2. **Article ratings**: how good and how important each article is.
3. **Talk page comments**: who discussed what on article talk pages.

This notebook we will:

- Load each dataset.
- Look at its main variables.
- Understand how to link them using `page_id`

In [1]:
import pandas as pd
import os

# ADJUST THE PATHS 

In [12]:
EDITS_PATH = "...data/revision_sample_800_articles.csv"
RATINGS_PATH = "...data/wp_medicine_ratings_800_sample.csv"
COMMENTS_PATH = "...data/all_comments_800_sample.csv"

Loading the data 

The data comes from the public Wikipedia dumps, which are archives that Wikipedia makes available online for free. These dumps contain the raw content of Wikipedia pages and metadata.

You can learn more about them here: https://meta.wikimedia.org/wiki/Data_dumps#:~:text=About%20Wikimedia%20Dumps&text=The%20dumps%20are%20used%20by,are%20produced%20once%20per%20month.

However, this is not the most important part of the project, the key focus is how to use this data

In [3]:
edits_df = pd.read_csv(EDITS_PATH)
ratings_df = pd.read_csv(RATINGS_PATH)
comments_df = pd.read_csv(COMMENTS_PATH)
edits_df.shape, ratings_df.shape, comments_df.shape

/var/folders/kr/vbpwwxg91gs4qts7462r9gsc0000gn/T/ipykernel_22269/3749685129.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  edits_df = pd.read_csv(EDITS_PATH)


((175365, 28), (800, 12), (3626, 9))

don't worry about the warning 

## 1. Edits dataset — `revision_sample_800_articles.csv`

**What is this file?**

Each row in this dataset represents **one edit** (one revision) to a medical article 

You can use it to study:

- Who edits which articles.
- How big their edits are.
- When edits happen 
- Whether edits are reverted (undone by another editor : meaning if a later editor cancels the edit)

**Key idea:** this file is our main source of **collaboration activity**.

In [4]:
edits_df.head()

,event_entity,event_timestamp,event_comment,event_user_text_historical,event_user_is_bot_by,event_user_is_anonymous,event_user_is_permanent,event_user_registration_timestamp,event_user_creation_timestamp,event_user_first_edit_timestamp,...,page_revision_count,page_seconds_since_previous_revision,revision_id,revision_parent_id,revision_minor_edit,revision_text_bytes,revision_text_bytes_diff,revision_is_identity_reverted,revision_is_identity_revert,revision_tags
0,revision,2003-02-19 21:45:36,"added ""trade-name"" Zyprexa for Olanzapine",Tzuhou,NaN,False,True,2003-02-19 20:11:40,NaN,2003-02-19 20:11:40,...,21.0,3380309.0,682836.0,682695.0,True,6117.0,10.0,False,False,NaN
1,revision,2003-02-19 22:19:45,NaN,Susan Mason,NaN,False,True,2003-02-18 12:44:29,NaN,2003-02-18 12:44:29,...,22.0,2049.0,682888.0,682836.0,False,6108.0,-9.0,False,False,NaN
2,revision,2003-02-19 22:35:36,NaN,Susan Mason,NaN,False,True,2003-02-18 12:44:29,NaN,2003-02-18 12:44:29,...,23.0,951.0,682890.0,682888.0,True,6110.0,2.0,False,False,NaN
3,revision,2003-02-19 22:36:03,NaN,Susan Mason,NaN,False,True,2003-02-18 12:44:29,NaN,2003-02-18 12:44:29,...,24.0,27.0,682892.0,682890.0,True,6124.0,14.0,False,False,NaN
4,revision,2003-02-19 22:36:22,NaN,Susan Mason,NaN,False,True,2003-02-18 12:44:29,NaN,2003-02-18 12:44:29,...,25.0,19.0,686312.0,682892.0,True,6043.0,-81.0,False,False,NaN


### 1.1 Main variables in the edits dataset

You don't have to every column

Here are the most important ones:

**About the edit (event)**

- `event_timestamp` – When the edit happened (date and time).
- `event_comment` – Short message written by the editor to explain the change.

**About the editor**

- `event_user_text_historical` – Username or IP address of the editor.
- `event_user_is_anonymous` – `True` if the editor is an IP (no account).
- `event_user_revision_count` – Total number of edits this editor has made.

**About the article**

- `page_id` – Unique numeric ID of the article.  
  This is the key that links all three datasets.
- `page_title` – Title of the article.
- `page_revision_count` – Total number of edits the article has received.

**About the revision (content change)**

- `revision_text_bytes` – Size of the article after this edit.
- `revision_text_bytes_diff` – Change in size for this edit  
  (positive = added text, negative = removed text).
- `revision_is_identity_revert` – `True` if this edit reverted another edit.
- `revision_is_identity_reverted` – `True` if this edit was later reverted.

You can use these variables to build:

- Editor–article networks.
- Editor–editor co-edit networks.
- Measures of edit size, conflict, and coordination.

In [5]:
print("Number of rows (edits):", len(edits_df))
print("Number of unique articles:", edits_df["page_id"].nunique())
print("Number of unique editors:", edits_df["event_user_text_historical"].nunique())

edits_df[["event_timestamp", "event_user_text_historical", "page_id", "revision_text_bytes_diff"]].head()

Number of rows (edits): 175365
Number of unique articles: 800
Number of unique editors: 53297


,event_timestamp,event_user_text_historical,page_id,revision_text_bytes_diff
0,2003-02-19 21:45:36,Tzuhou,2870,10.0
1,2003-02-19 22:19:45,Susan Mason,2870,-9.0
2,2003-02-19 22:35:36,Susan Mason,2870,2.0
3,2003-02-19 22:36:03,Susan Mason,2870,14.0
4,2003-02-19 22:36:22,Susan Mason,2870,-81.0


## 2. Article ratings dataset — `wp_medicine_ratings_800_sample.csv`

**What is this file?**

Each row in this dataset represents **one article**.  

It tells you how articles has assessed:

- The **quality** of the article (e.g., Stub, Start, C, B, GA, FA). Here is a link to help you understand what each rating means : https://en.wikipedia.org/wiki/Wikipedia:Content_assessment. For example,  FA means : featured article and it's the best possible rating. 

- The **importance** of the article (e.g., Low, Mid, High, Top). Importance is how much of a priority an article is for a given project. For example :  Low-importance → The article is not a priority for that WikiProject. It covers a topic that is relevant but not central.

Mid-importance → Moderately important; useful or notable within the project.

High-importance → Very important; core topic

This helps you link collaboration patterns to how good/important the article is.

In [6]:
ratings_df.head()

,article,importance,importance_updated,quality,quality_updated,article_history_link,article_link,article_talk,article_talk_link,title,page_id,ns
0,William S. Sadler,Low-Class,2012-03-14T18:16:30Z,FA-Class,2012-06-24T00:33:08Z,https://en.wikipedia.org/w/index.php?title=Wil...,https://en.wikipedia.org/w/index.php?title=Wil...,Talk:William S. Sadler,https://en.wikipedia.org/w/index.php?title=Tal...,William S. Sadler,2113994,0.0
1,List of books bound in human skin,Low-Class,2023-09-03T04:02:48Z,FL-Class,2023-10-09T00:25:40Z,https://en.wikipedia.org/w/index.php?title=Lis...,https://en.wikipedia.org/w/index.php?title=Lis...,Talk:List of books bound in human skin,https://en.wikipedia.org/w/index.php?title=Tal...,List of books bound in human skin,74680914,0.0
2,Hemothorax,Mid-Class,2007-10-11T21:33:48Z,GA-Class,2019-04-25T16:47:04Z,https://en.wikipedia.org/w/index.php?title=Hem...,https://en.wikipedia.org/w/index.php?title=Hem...,Talk:Hemothorax,https://en.wikipedia.org/w/index.php?title=Tal...,Hemothorax,2166658,0.0
3,Histamine N-methyltransferase,Mid-Class,2023-12-01T11:35:19Z,GA-Class,2024-03-26T19:05:02Z,https://en.wikipedia.org/w/index.php?title=His...,https://en.wikipedia.org/w/index.php?title=His...,Talk:Histamine N-methyltransferase,https://en.wikipedia.org/w/index.php?title=Tal...,Histamine N-methyltransferase,11472211,0.0
4,"2,3,7,8-Tetrachlorodibenzodioxin",Low-Class,2010-02-09T21:59:21Z,GA-Class,2012-10-14T02:09:51Z,https://en.wikipedia.org/w/index.php?title=2%2...,https://en.wikipedia.org/w/index.php?title=2%2...,"Talk:2,3,7,8-Tetrachlorodibenzodioxin",https://en.wikipedia.org/w/index.php?title=Tal...,"2,3,7,8-Tetrachlorodibenzodioxin",23789332,0.0


### 2.1 Main variables in the ratings dataset

- `page_id` – Article ID (same as in the edits dataset).  
- `article` / `title` – Title of the article.
- `quality` – Quality rating (e.g., `FA-Class`, `GA-Class`, `B-Class`, `Start-Class`, `Stub-Class`).
- `quality_updated` – When the quality rating was last updated.
- `importance` – Importance rating inside WikiProject Medicine (`Top-Class`, `High-Class`, `Mid-Class`, etc.).
- `importance_updated` – When the importance rating was last updated.

Other columns (`article_link`, `article_history_link`, `article_talk_link`) are URLs and not essential for the analysis.

In [7]:
print("Rows in ratings_df:", len(ratings_df))
print("Unique articles in ratings_df:", ratings_df["page_id"].nunique())

# How many of our 800 sampled articles have ratings?
common_ids = set(edits_df["page_id"].unique()) & set(ratings_df["page_id"].unique())
print("Articles present in both edits and ratings:", len(common_ids))

Rows in ratings_df: 800
Unique articles in ratings_df: 800
Articles present in both edits and ratings: 800


## 3. Talk comments dataset — `all_comments_800_sample.csv`

**What is this file?**

Each row in this dataset represents **one comment** on an article's talk page.

On Wikipedia, each article has a **talk page** where editors discuss:

- Proposed changes
- Disagreements
- Sources
- Structure of the article

This file lets you study **who talks to whom, and when**, about which article.

In [8]:
comments_df.head()

,id,section,user,timestamp,comment,parent_id,indent_level,article,page_id
0,2113994_53cde6_1,(untitled),T. Anthony,"08:51, 6 October 2005",Something about this strikes me as vaguely POV...,NaN,0,William S. Sadler,2113994
1,2113994_6fbd41_1,Improvements,Edivorce,"16:37, 16 December 2005",A few months ago I made some minor edits to th...,NaN,0,William S. Sadler,2113994
2,2113994_6fbd41_2,Improvements,96.239.135.225,"22:41, 21 August 2008",Sadler published a book titled Long Heads and ...,NaN,0,William S. Sadler,2113994
3,2113994_cf045e_1,More improvements needed,ScienceStorm,"04:36, 10 July 2009",In the text:\n“Sadler did not adhere to purely...,NaN,0,William S. Sadler,2113994
4,2113994_b7525f_1,Inadequate Article,Ldmjr,"08:15, 16 August 2009",This article serious misrepresents the man and...,NaN,0,William S. Sadler,2113994


### 3.1 Main variables in the talk comments dataset

- `page_id` – Article ID (same as in the edits and ratings datasets).
- `article` – Article title.

- `id` – Unique ID of the comment.
- `section` – Title of the discussion section on the talk page.
- `user` – Username of the person who wrote the comment.
- `timestamp` – When the comment was posted (wiki time format).
- `comment` – Text of the comment.

- `parent_id` – If this is a reply, the `id` of the comment it replies to.  
  If empty, it starts a new thread.
- `indent_level` – Depth of the reply (0 = main comment, 1 = reply, 2 = reply to reply, etc.).

This dataset is useful if you want to look at:

- Discussion intensity per article.
- Who interacts with whom on talk pages.
- Relation between talk activity, edit patterns, and article quality.

In [9]:
print("Rows in comments_df:", len(comments_df))
print("Unique articles with comments:", comments_df["page_id"].nunique())

common_ids_comments = set(edits_df["page_id"].unique()) & set(comments_df["page_id"].unique())
print("Articles present in both edits and comments:", len(common_ids_comments))

Rows in comments_df: 3626
Unique articles with comments: 417
Articles present in both edits and comments: 417


## 4. How the datasets fit together

All three datasets share the same key: **`page_id`**.

- The **edits dataset** (`revision_sample_800_articles.csv`) tells you:
  - Who edited which article, when, and how much they changed.

- The **ratings dataset** (`wp_medicine_ratings_800_sample.csv`) tells you:
  - How good and important each article is (quality and importance).

- The **talk comments dataset** (`all_comments_800_sample.csv`) tells you:
  - Who discussed which article on the talk page, and what they said.

You will use `page_id` to join these datasets when/ if needed.

In [10]:
edits_with_ratings = edits_df.merge(
    ratings_df[["page_id", "quality", "importance"]],
    on="page_id",
    how="left"
)

edits_with_ratings[["page_id", "page_title", "event_user_text_historical", "quality", "importance"]].head()

,page_id,page_title,event_user_text_historical,quality,importance
0,2870,Antipsychotic,Tzuhou,B-Class,High-Class
1,2870,Antipsychotic,Susan Mason,B-Class,High-Class
2,2870,Antipsychotic,Susan Mason,B-Class,High-Class
3,2870,Antipsychotic,Susan Mason,B-Class,High-Class
4,2870,Antipsychotic,Susan Mason,B-Class,High-Class


This is just one example of a merge, you'll probably need to change it. The goal is to show you that you that it's with page_id